## setting up

In [1]:
!pip install linearmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 13.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [linearmodels]


In [30]:
#package
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS
from linearmodels.iv import IV2SLS

In [40]:
# environmental info
import sys
import pandas as pd
import numpy as np
import matplotlib
import statsmodels
import linearmodels

# Print package versions
print(f"Python:       {sys.version.split()[0]}")
print(f"pandas:       {pd.__version__}")
print(f"numpy:        {np.__version__}")
print(f"matplotlib:   {matplotlib.__version__}")
print(f"statsmodels:  {statsmodels.__version__}")
print(f"linearmodels: {linearmodels.__version__}")

Python:       3.13.18
pandas:       2.2.3
numpy:        2.4.6
matplotlib:   3.10.0
statsmodels:  0.15.0
linearmodels: 7.1


In [3]:
#set current directrory
# os.chdir("")

In [4]:
#read data
#※data ignored as a public repository
df_a = pd.read_stata("data/SLP75_81.DTA")
df_b = pd.read_stata("data/ajrdata.dta")

## A

### Data preparation

In [5]:
df_a.columns

Index(['age75', 'educ75', 'educ81', 'gdhlth75', 'gdhlth81', 'male', 'marr75',
       'marr81', 'slpnap75', 'slpnap81', 'totwrk75', 'totwrk81', 'yngkid75',
       'yngkid81', 'ceduc', 'cgdhlth', 'cmarr', 'cslpnap', 'ctotwrk',
       'cyngkid'],
      dtype='object')

In [6]:
#create age81
df_a["age81"] = df_a["age75"] + 6
df_a["age81"].describe()

,age81
count,239.000000
mean,45.012552
std,11.066826
min,29.000000
25%,36.000000
50%,42.000000
75%,53.000000
max,71.000000


In [7]:
# drop incomplete observations
level_cols = [
    "age75",
    "educ75", "educ81",
    "gdhlth75", "gdhlth81",
    "marr75", "marr81",
    "slpnap75", "slpnap81",
    "totwrk75", "totwrk81",
    "yngkid75", "yngkid81",
]
c_cols = ["cslpnap", "ctotwrk", "cmarr", "ceduc", "cgdhlth", "cyngkid"]

df_a = df_a.dropna(subset=level_cols + c_cols)

In [8]:
#generate_id
df_a["id"] = df_a.index + 1

#variables
stubs = ["age", "educ", "gdhlth", "marr", "slpnap", "totwrk", "yngkid"]

#create long data
df_a_panel = pd.wide_to_long(
    df_a,
    stubnames=stubs,
    i="id",
    j="year",
    sep="",
    suffix=r"\d+",
).reset_index()

#create year variables and generate index (keep id and year as columns too)
df_a_panel["year"] = df_a_panel["year"] + 1900
df_a_panel = (
    df_a_panel.sort_values(["id", "year"])
    .set_index(["id", "year"], drop=False)
)

#filtered complete data
complete = (
    df_a_panel[stubs]
    .notna()
    .all(axis=1)
    .groupby(level="id")
    .transform("all")
)
df_a_panel = df_a_panel[complete]

### A1. First differences.

In [9]:
#regression
model_a1 = smf.ols(
    "cslpnap ~ ctotwrk + cmarr + ceduc + cgdhlth + cyngkid",
    data=df_a,
).fit()

print(model_a1.summary())

                            OLS Regression Results                            
Dep. Variable:                cslpnap   R-squared:                       0.150
Model:                            OLS   Adj. R-squared:                  0.131
Method:                 Least Squares   F-statistic:                     8.191
Date:                Thu, 05 Nov 2026   Prob (F-statistic):           3.83e-07
Time:                        00:59:49   Log-Likelihood:                -1864.4
No. Observations:                 239   AIC:                             3741.
Df Residuals:                     233   BIC:                             3762.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    -92.6340     45.866     -2.020      0.0

In [10]:
#write to tabele
labels = {
    "ctotwrk": r"$\Delta$ Work (min/week)",
    "cmarr": r"$\Delta$ Married",
    "ceduc": r"$\Delta$ Education",
    "cgdhlth": r"$\Delta$ Good health",
    "cyngkid": r"$\Delta$ Child under 3",
    "Intercept": "Constant",
}

rows = []
for var, label in labels.items():
    rows.append([label, f"{model_a1.params[var]:.4f}"])
    rows.append(["", f"({model_a1.bse[var]:.4f})"])
rows.append(["N", f"{int(model_a1.nobs)}"])

table_a1 = pd.DataFrame(rows, columns=["", r"$\Delta$ Sleep and naps"])

print(table_a1.to_latex(index=False, escape=False))

\begin{tabular}{ll}
\toprule
 & $\Delta$ Sleep and naps \\
\midrule
$\Delta$ Work (min/week) & -0.2267 \\
 & (0.0361) \\
$\Delta$ Married & 104.2139 \\
 & (92.8554) \\
$\Delta$ Education & -0.0245 \\
 & (48.7594) \\
$\Delta$ Good health & 87.5778 \\
 & (76.5991) \\
$\Delta$ Child under 3 & 94.6654 \\
 & (87.6525) \\
Constant & -92.6340 \\
 & (45.8659) \\
N & 239 \\
\bottomrule
\end{tabular}



### A2. Fixed effects: individual indicators and the within estimator

In [11]:
#regression LSDV
formula = "slpnap ~ totwrk + age + educ + marr + gdhlth + yngkid + C(id)"

lsdv_noyear = smf.ols(formula, data=df_a_panel).fit()

print(lsdv_noyear.summary())

                            OLS Regression Results                            
Dep. Variable:                 slpnap   R-squared:                       0.682
Model:                            OLS   Adj. R-squared:                  0.349
Method:                 Least Squares   F-statistic:                     2.046
Date:                Thu, 05 Nov 2026   Prob (F-statistic):           2.54e-08
Time:                        00:59:49   Log-Likelihood:                -3397.4
No. Observations:                 478   AIC:                             7285.
Df Residuals:                     233   BIC:                             8306.
Df Model:                         244                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     5467.9575    826.863      6.613   

In [12]:
#regression with year fix
formula = "slpnap ~ totwrk + age + educ + marr + gdhlth + yngkid + C(id)"

lsdv_year = smf.ols(formula + " + C(year)", data=df_a_panel).fit()

print(lsdv_year.summary())

/tmp/ipykernel_3280/2041770156.py:4: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  lsdv_year = smf.ols(formula + " + C(year)", data=df_a_panel).fit()


                            OLS Regression Results                            
Dep. Variable:                 slpnap   R-squared:                       0.682
Model:                            OLS   Adj. R-squared:                  0.349
Method:                 Least Squares   F-statistic:                     2.046
Date:                Thu, 05 Nov 2026   Prob (F-statistic):           2.54e-08
Time:                        00:59:50   Log-Likelihood:                -3397.4
No. Observations:                 478   AIC:                             7285.
Df Residuals:                     233   BIC:                             8306.
Df Model:                         244                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept        2600.5304    669.603     

In [13]:
#regression within
regressors = "totwrk + age + educ + marr + gdhlth + yngkid"

within_noyear = PanelOLS.from_formula(
    f"slpnap ~ 1 + {regressors} + EntityEffects", data=df_a_panel
).fit()

print(within_noyear.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:                 slpnap   R-squared:                        0.1552
Estimator:                   PanelOLS   R-squared (Between):             -0.1987
No. Observations:                 478   R-squared (Within):               0.1552
Date:                Thu, Nov 05 2026   R-squared (Overall):             -0.0654
Time:                        00:59:50   Log-likelihood                   -3397.4
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      7.1344
Entities:                         239   P-value                           0.0000
Avg Obs:                       2.0000   Distribution:                   F(6,233)
Min Obs:                       2.0000                                           
Max Obs:                       2.0000   F-statistic (robust):             7.1344
                            

In [14]:
#regression within with year fix
within_year = PanelOLS.from_formula(
    f"slpnap ~ 1 + {regressors} + C(year) + EntityEffects", data=df_a_panel,
    drop_absorbed=True,
).fit()

print(within_year.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                 slpnap   R-squared:                        0.1552
Estimator:                   PanelOLS   R-squared (Between):             -0.1987
No. Observations:                 478   R-squared (Within):               0.1552
Date:                Thu, Nov 05 2026   R-squared (Overall):             -0.0654
Time:                        00:59:51   Log-likelihood                   -3397.4
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      7.1344
Entities:                         239   P-value                           0.0000
Avg Obs:                       2.0000   Distribution:                   F(6,233)
Min Obs:                       2.0000                                           
Max Obs:                       2.0000   F-statistic (robust):             7.1344
                            

/tmp/ipykernel_3280/3900504997.py:5: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

C(year)[T.1981]

  ).fit()


In [15]:
#write to table
models = {
    "(1) LSDV": lsdv_noyear,
    "(2) LSDV": lsdv_year,
    "(3) Within": within_noyear,
    "(4) Within": within_year,
}

def coef_se(m):
    se = m.bse if hasattr(m, "bse") else m.std_errors
    return m.params, se

coef_row, se_row = ["Work (min/week)"], [""]
for m in models.values():
    params, se = coef_se(m)
    coef_row.append(f"{params['totwrk']:.4f}")
    se_row.append(f"({se['totwrk']:.4f})")

circle, cross = r"$\bigcirc$", r"$\times$"

rows = [
    coef_row,
    se_row,
    ["Age"] + [circle] * 4,
    ["Education"] + [circle] * 4,
    ["Married"] + [circle] * 4,
    ["Good health"] + [circle] * 4,
    ["Child under 3"] + [circle] * 4,
    ["Person effects"] + [circle] * 4,
    ["Year indicator", cross, circle, cross, circle],
    ["N"] + [f"{int(m.nobs)}" for m in models.values()],
]

table_a2 = pd.DataFrame(rows, columns=["Sleep and naps"] + list(models))

print(table_a2.to_latex(
    index=False,
    escape=False,
    caption="Fixed-effects estimates of sleep and naps: LSDV and within estimators",
))

\begin{table}
\caption{Fixed-effects estimates of sleep and naps: LSDV and within estimators}
\begin{tabular}{lllll}
\toprule
Sleep and naps & (1) LSDV & (2) LSDV & (3) Within & (4) Within \\
\midrule
Work (min/week) & -0.2267 & -0.2267 & -0.2267 & -0.2267 \\
 & (0.0361) & (0.0361) & (0.0361) & (0.0361) \\
Age & $\bigcirc$ & $\bigcirc$ & $\bigcirc$ & $\bigcirc$ \\
Education & $\bigcirc$ & $\bigcirc$ & $\bigcirc$ & $\bigcirc$ \\
Married & $\bigcirc$ & $\bigcirc$ & $\bigcirc$ & $\bigcirc$ \\
Good health & $\bigcirc$ & $\bigcirc$ & $\bigcirc$ & $\bigcirc$ \\
Child under 3 & $\bigcirc$ & $\bigcirc$ & $\bigcirc$ & $\bigcirc$ \\
Person effects & $\bigcirc$ & $\bigcirc$ & $\bigcirc$ & $\bigcirc$ \\
Year indicator & $\times$ & $\bigcirc$ & $\times$ & $\bigcirc$ \\
N & 478 & 478 & 478 & 478 \\
\bottomrule
\end{tabular}
\end{table}



### A3.Uncertainty and causal interpretation.

In [16]:
#regression within with year fix, cluster-robust SE by person
within_year_cl = PanelOLS.from_formula(
    f"slpnap ~ 1 + {regressors} + C(year) + EntityEffects", data=df_a_panel,
    drop_absorbed=True,
).fit(
    cov_type="clustered",
    cluster_entity=True,
    debiased=True,
    group_debias=True,
    count_effects=False,
)

print(within_year_cl.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                 slpnap   R-squared:                        0.1552
Estimator:                   PanelOLS   R-squared (Between):             -0.1987
No. Observations:                 478   R-squared (Within):               0.1552
Date:                Thu, Nov 05 2026   R-squared (Overall):             -0.0654
Time:                        00:59:51   Log-likelihood                   -3397.4
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      7.1344
Entities:                         239   P-value                           0.0000
Avg Obs:                       2.0000   Distribution:                   F(6,233)
Min Obs:                       2.0000                                           
Max Obs:                       2.0000   F-statistic (robust):             7.5566
                            

/tmp/ipykernel_3280/3382852156.py:5: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

C(year)[T.1981]

  ).fit(


In [17]:
#table
models = {
    "(1) Within": within_year,
    "(2) Within": within_year_cl,
}

coef_row, se_row = ["Work (min/week)"], [""]
for m in models.values():
    coef_row.append(f"{m.params['totwrk']:.4f}")
    se_row.append(f"({m.std_errors['totwrk']:.4f})")

circle = r"$\bigcirc$"

rows = [
    coef_row,
    se_row,
    ["Age"] + [circle] * 2,
    ["Education"] + [circle] * 2,
    ["Married"] + [circle] * 2,
    ["Good health"] + [circle] * 2,
    ["Child under 3"] + [circle] * 2,
    ["Person effects"] + [circle] * 2,
    ["Year indicator"] + [circle] * 2,
    ["Standard errors", "Conventional", "Clustered by person"],
    ["Clusters", "", f"{df_a_panel['id'].nunique()}"],
    ["N"] + [f"{int(m.nobs)}" for m in models.values()],
]

table_a3 = pd.DataFrame(rows, columns=["Sleep and naps"] + list(models))

print(table_a3.to_latex(
    index=False,
    escape=False,
    caption="Within estimates of sleep and naps: conventional and cluster-robust standard errors",
))

\begin{table}
\caption{Within estimates of sleep and naps: conventional and cluster-robust standard errors}
\begin{tabular}{lll}
\toprule
Sleep and naps & (1) Within & (2) Within \\
\midrule
Work (min/week) & -0.2267 & -0.2267 \\
 & (0.0361) & (0.0365) \\
Age & $\bigcirc$ & $\bigcirc$ \\
Education & $\bigcirc$ & $\bigcirc$ \\
Married & $\bigcirc$ & $\bigcirc$ \\
Good health & $\bigcirc$ & $\bigcirc$ \\
Child under 3 & $\bigcirc$ & $\bigcirc$ \\
Person effects & $\bigcirc$ & $\bigcirc$ \\
Year indicator & $\bigcirc$ & $\bigcirc$ \\
Standard errors & Conventional & Clustered by person \\
Clusters &  & 239 \\
N & 478 & 478 \\
\bottomrule
\end{tabular}
\end{table}



## B

### B1. OLS.

In [21]:
#check clumns
df_b.columns

Index(['shortnam', 'africa', 'lat_abst', 'rich4', 'avexpr', 'logpgp95',
       'logem4', 'asia', 'loghjypl', 'other_cont'],
      dtype='object')

In [24]:
#check na
df_b["logem4"].isna().sum()

np.int64(0)

In [25]:
#Table 2 column (2)
ols_2 = smf.ols("logpgp95 ~ avexpr", data=df_b).fit()
print(ols_2.summary())

                            OLS Regression Results                            
Dep. Variable:               logpgp95   R-squared:                       0.540
Model:                            OLS   Adj. R-squared:                  0.533
Method:                 Least Squares   F-statistic:                     72.82
Date:                Thu, 05 Nov 2026   Prob (F-statistic):           4.72e-12
Time:                        01:16:07   Log-Likelihood:                -68.168
No. Observations:                  64   AIC:                             140.3
Df Residuals:                      62   BIC:                             144.7
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      4.6604      0.409     11.408      0.0

In [26]:
#Table 2 column (5)
ols_5 = smf.ols("logpgp95 ~ avexpr + lat_abst", data=df_b).fit()
print(ols_5.summary())

                            OLS Regression Results                            
Dep. Variable:               logpgp95   R-squared:                       0.574
Model:                            OLS   Adj. R-squared:                  0.561
Method:                 Least Squares   F-statistic:                     41.18
Date:                Thu, 05 Nov 2026   Prob (F-statistic):           4.80e-12
Time:                        01:16:35   Log-Likelihood:                -65.681
No. Observations:                  64   AIC:                             137.4
Df Residuals:                      61   BIC:                             143.8
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      4.7281      0.397     11.900      0.0

In [27]:
#Table 2 column (6)
ols_6 = smf.ols(
    "logpgp95 ~ avexpr + lat_abst + asia + africa + other_cont", data=df_b
).fit()
print(ols_6.summary())

                            OLS Regression Results                            
Dep. Variable:               logpgp95   R-squared:                       0.714
Model:                            OLS   Adj. R-squared:                  0.689
Method:                 Least Squares   F-statistic:                     28.95
Date:                Thu, 05 Nov 2026   Prob (F-statistic):           1.33e-14
Time:                        01:16:45   Log-Likelihood:                -52.979
No. Observations:                  64   AIC:                             118.0
Df Residuals:                      58   BIC:                             130.9
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      5.7367      0.398     14.407      0.0

In [28]:
#Table 2 column (8)
ols_8 = smf.ols("loghjypl ~ avexpr", data=df_b).fit()
print(ols_8.summary())

                            OLS Regression Results                            
Dep. Variable:               loghjypl   R-squared:                       0.486
Model:                            OLS   Adj. R-squared:                  0.477
Method:                 Least Squares   F-statistic:                     55.83
Date:                Thu, 05 Nov 2026   Prob (F-statistic):           4.29e-10
Time:                        01:16:55   Log-Likelihood:                -64.555
No. Observations:                  61   AIC:                             133.1
Df Residuals:                      59   BIC:                             137.3
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -4.9117      0.409    -12.017      0.0

In [29]:
#create tabele
#table
labels = {
    "avexpr": "Average protection against expropriation risk, 1985--1995",
    "lat_abst": "Latitude",
    "asia": "Asia dummy",
    "africa": "Africa dummy",
    "other_cont": "``Other'' continent dummy",
}
models = {
    "(2)": ols_2,
    "(5)": ols_5,
    "(6)": ols_6,
    "(8)": ols_8,
}

rows = []
for var, label in labels.items():
    coef_row, se_row = [label], [""]
    for m in models.values():
        if var in m.params.index:
            coef_row.append(f"{m.params[var]:.2f}")
            se_row.append(f"({m.bse[var]:.2f})")
        else:
            coef_row.append("")
            se_row.append("")
    rows += [coef_row, se_row]

rows.append(["Dependent variable"] + ["Log GDP per capita, 1995"] * 3 + ["Log output per worker, 1988"])
rows.append(["N"] + [f"{int(m.nobs)}" for m in models.values()])

table_b1 = pd.DataFrame(rows, columns=[""] + list(models))

print(table_b1.to_latex(
    index=False,
    escape=False,
    caption="OLS regressions: replication of Table 2, columns (2), (5), (6), and (8)",
))

\begin{table}
\caption{OLS regressions: replication of Table 2, columns (2), (5), (6), and (8)}
\begin{tabular}{lllll}
\toprule
 & (2) & (5) & (6) & (8) \\
\midrule
Average protection against expropriation risk, 1985--1995 & 0.52 & 0.47 & 0.40 & 0.46 \\
 & (0.06) & (0.06) & (0.06) & (0.06) \\
Latitude &  & 1.58 & 0.88 &  \\
 &  & (0.71) & (0.63) &  \\
Asia dummy &  &  & -0.58 &  \\
 &  &  & (0.23) &  \\
Africa dummy &  &  & -0.88 &  \\
 &  &  & (0.17) &  \\
``Other'' continent dummy &  &  & 0.11 &  \\
 &  &  & (0.38) &  \\
Dependent variable & Log GDP per capita, 1995 & Log GDP per capita, 1995 & Log GDP per capita, 1995 & Log output per worker, 1988 \\
N & 64 & 64 & 64 & 61 \\
\bottomrule
\end{tabular}
\end{table}



### B2. Two-stage least squares (2SLS)

In [31]:
#Table 4 Panel A column (1)
iv_1 = IV2SLS.from_formula(
    "logpgp95 ~ 1 + [avexpr ~ logem4]", data=df_b
).fit(cov_type="unadjusted")
print(iv_1.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:               logpgp95   R-squared:                      0.1870
Estimator:                    IV-2SLS   Adj. R-squared:                 0.1739
No. Observations:                  64   F-statistic:                    37.568
Date:                Thu, Nov 05 2026   P-value (F-stat)                0.0000
Time:                        01:42:09   Distribution:                  chi2(1)
Cov. Estimator:            unadjusted                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
Intercept      1.9097     1.0106     1.8897     0.0588     -0.0710      3.8903
avexpr         0.9443     0.1541     6.1293     0.00

In [32]:

#Table 4 Panel A column (2)
iv_2 = IV2SLS.from_formula(
    "logpgp95 ~ 1 + lat_abst + [avexpr ~ logem4]", data=df_b
).fit(cov_type="unadjusted")
print(iv_2.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:               logpgp95   R-squared:                      0.1025
Estimator:                    IV-2SLS   Adj. R-squared:                 0.0730
No. Observations:                  64   F-statistic:                    35.683
Date:                Thu, Nov 05 2026   P-value (F-stat)                0.0000
Time:                        01:42:22   Distribution:                  chi2(2)
Cov. Estimator:            unadjusted                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
Intercept      1.6918     1.2623     1.3402     0.1802     -0.7823      4.1659
lat_abst      -0.6472     1.3035    -0.4965     0.61

In [33]:
#Table 4 Panel A column (8)
iv_8 = IV2SLS.from_formula(
    "logpgp95 ~ 1 + lat_abst + asia + africa + other_cont + [avexpr ~ logem4]",
    data=df_b,
).fit(cov_type="unadjusted")
print(iv_8.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:               logpgp95   R-squared:                      0.0108
Estimator:                    IV-2SLS   Adj. R-squared:                -0.0745
No. Observations:                  64   F-statistic:                    37.778
Date:                Thu, Nov 05 2026   P-value (F-stat)                0.0000
Time:                        01:42:29   Distribution:                  chi2(5)
Cov. Estimator:            unadjusted                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
Intercept      1.4405     2.7032     0.5329     0.5941     -3.8577      6.7386
lat_abst      -1.1782     1.6711    -0.7050     0.48

In [34]:
#Table 4 Panel A column (9)
iv_9 = IV2SLS.from_formula(
    "loghjypl ~ 1 + [avexpr ~ logem4]", data=df_b
).fit(cov_type="unadjusted")
print(iv_9.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:               loghjypl   R-squared:                     -0.1518
Estimator:                    IV-2SLS   Adj. R-squared:                -0.1714
No. Observations:                  61   F-statistic:                    34.049
Date:                Thu, Nov 05 2026   P-value (F-stat)                0.0000
Time:                        01:42:35   Distribution:                  chi2(1)
Cov. Estimator:            unadjusted                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
Intercept     -8.3229     1.1030    -7.5456     0.0000     -10.485     -6.1610
avexpr         0.9808     0.1681     5.8352     0.00

/usr/local/lib/python3.13/dist-packages/linearmodels/iv/model.py:789: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(


In [35]:
#create table
#table
labels = {
    "avexpr": "Average protection against expropriation risk, 1985--1995",
    "lat_abst": "Latitude",
    "asia": "Asia dummy",
    "africa": "Africa dummy",
    "other_cont": "``Other'' continent dummy",
}
models = {
    "(1)": iv_1,
    "(2)": iv_2,
    "(8)": iv_8,
    "(9)": iv_9,
}

rows = []
for var, label in labels.items():
    coef_row, se_row = [label], [""]
    for m in models.values():
        if var in m.params.index:
            coef_row.append(f"{m.params[var]:.2f}")
            se_row.append(f"({m.std_errors[var]:.2f})")
        else:
            coef_row.append("")
            se_row.append("")
    rows += [coef_row, se_row]

rows.append(["Dependent variable"] + ["Log GDP per capita, 1995"] * 3 + ["Log output per worker, 1988"])
rows.append(["N"] + [f"{int(m.nobs)}" for m in models.values()])

table_b2 = pd.DataFrame(rows, columns=[""] + list(models))

print(table_b2.to_latex(
    index=False,
    escape=False,
    caption="Two-stage least squares: replication of Table 4, Panel A, columns (1), (2), (8), and (9)",
))

\begin{table}
\caption{Two-stage least squares: replication of Table 4, Panel A, columns (1), (2), (8), and (9)}
\begin{tabular}{lllll}
\toprule
 & (1) & (2) & (8) & (9) \\
\midrule
Average protection against expropriation risk, 1985--1995 & 0.94 & 1.00 & 1.11 & 0.98 \\
 & (0.15) & (0.22) & (0.44) & (0.17) \\
Latitude &  & -0.65 & -1.18 &  \\
 &  & (1.30) & (1.67) &  \\
Asia dummy &  &  & -1.05 &  \\
 &  &  & (0.50) &  \\
Africa dummy &  &  & -0.44 &  \\
 &  &  & (0.40) &  \\
``Other'' continent dummy &  &  & -0.99 &  \\
 &  &  & (0.95) &  \\
Dependent variable & Log GDP per capita, 1995 & Log GDP per capita, 1995 & Log GDP per capita, 1995 & Log output per worker, 1988 \\
N & 64 & 64 & 64 & 61 \\
\bottomrule
\end{tabular}
\end{table}



### B3. Robust standard errors.

In [36]:
#recalcu B1 with robust
#Table 2 column (2), robust SE
ols_2_r = smf.ols("logpgp95 ~ avexpr", data=df_b).fit(cov_type="HC1")
print(ols_2_r.summary())

#Table 2 column (5), robust SE
ols_5_r = smf.ols("logpgp95 ~ avexpr + lat_abst", data=df_b).fit(cov_type="HC1")
print(ols_5_r.summary())

#Table 2 column (6), robust SE
ols_6_r = smf.ols(
    "logpgp95 ~ avexpr + lat_abst + asia + africa + other_cont", data=df_b
).fit(cov_type="HC1")
print(ols_6_r.summary())

#Table 2 column (8), robust SE
ols_8_r = smf.ols("loghjypl ~ avexpr", data=df_b).fit(cov_type="HC1")
print(ols_8_r.summary())

                            OLS Regression Results                            
Dep. Variable:               logpgp95   R-squared:                       0.540
Model:                            OLS   Adj. R-squared:                  0.533
Method:                 Least Squares   F-statistic:                     109.4
Date:                Thu, 05 Nov 2026   Prob (F-statistic):           2.57e-15
Time:                        02:39:19   Log-Likelihood:                -68.168
No. Observations:                  64   AIC:                             140.3
Df Residuals:                      62   BIC:                             144.7
Df Model:                           1                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      4.6604      0.320     14.558      0.0

In [37]:
#table
labels = {
    "avexpr": "Average protection against expropriation risk, 1985--1995",
    "lat_abst": "Latitude",
    "asia": "Asia dummy",
    "africa": "Africa dummy",
    "other_cont": "``Other'' continent dummy",
}
models = {
    "(2)": ols_2_r,
    "(5)": ols_5_r,
    "(6)": ols_6_r,
    "(8)": ols_8_r,
}

rows = []
for var, label in labels.items():
    coef_row, se_row = [label], [""]
    for m in models.values():
        if var in m.params.index:
            coef_row.append(f"{m.params[var]:.2f}")
            se_row.append(f"({m.bse[var]:.2f})")
        else:
            coef_row.append("")
            se_row.append("")
    rows += [coef_row, se_row]

rows.append(["Dependent variable"] + ["Log GDP per capita, 1995"] * 3 + ["Log output per worker, 1988"])
rows.append(["Standard errors"] + ["Robust (HC1)"] * 4)
rows.append(["N"] + [f"{int(m.nobs)}" for m in models.values()])

table_b3_ols = pd.DataFrame(rows, columns=[""] + list(models))

print(table_b3_ols.to_latex(
    index=False,
    escape=False,
    caption="OLS regressions with heteroskedasticity-robust standard errors: Table 2, columns (2), (5), (6), and (8)",
))

\begin{table}
\caption{OLS regressions with heteroskedasticity-robust standard errors: Table 2, columns (2), (5), (6), and (8)}
\begin{tabular}{lllll}
\toprule
 & (2) & (5) & (6) & (8) \\
\midrule
Average protection against expropriation risk, 1985--1995 & 0.52 & 0.47 & 0.40 & 0.46 \\
 & (0.05) & (0.06) & (0.06) & (0.05) \\
Latitude &  & 1.58 & 0.88 &  \\
 &  & (0.65) & (0.61) &  \\
Asia dummy &  &  & -0.58 &  \\
 &  &  & (0.30) &  \\
Africa dummy &  &  & -0.88 &  \\
 &  &  & (0.16) &  \\
``Other'' continent dummy &  &  & 0.11 &  \\
 &  &  & (0.22) &  \\
Dependent variable & Log GDP per capita, 1995 & Log GDP per capita, 1995 & Log GDP per capita, 1995 & Log output per worker, 1988 \\
Standard errors & Robust (HC1) & Robust (HC1) & Robust (HC1) & Robust (HC1) \\
N & 64 & 64 & 64 & 61 \\
\bottomrule
\end{tabular}
\end{table}



In [38]:
#recalcu B2 with robust
#Table 4 Panel A column (1), robust SE
iv_1_r = IV2SLS.from_formula(
    "logpgp95 ~ 1 + [avexpr ~ logem4]", data=df_b
).fit(cov_type="robust")
print(iv_1_r.summary)

#Table 4 Panel A column (2), robust SE
iv_2_r = IV2SLS.from_formula(
    "logpgp95 ~ 1 + lat_abst + [avexpr ~ logem4]", data=df_b
).fit(cov_type="robust")
print(iv_2_r.summary)

#Table 4 Panel A column (8), robust SE
iv_8_r = IV2SLS.from_formula(
    "logpgp95 ~ 1 + lat_abst + asia + africa + other_cont + [avexpr ~ logem4]",
    data=df_b,
).fit(cov_type="robust")
print(iv_8_r.summary)

#Table 4 Panel A column (9), robust SE
iv_9_r = IV2SLS.from_formula(
    "loghjypl ~ 1 + [avexpr ~ logem4]", data=df_b
).fit(cov_type="robust")
print(iv_9_r.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:               logpgp95   R-squared:                      0.1870
Estimator:                    IV-2SLS   Adj. R-squared:                 0.1739
No. Observations:                  64   F-statistic:                    28.754
Date:                Thu, Nov 05 2026   P-value (F-stat)                0.0000
Time:                        02:39:57   Distribution:                  chi2(1)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
Intercept      1.9097     1.1740     1.6267     0.1038     -0.3912      4.2106
avexpr         0.9443     0.1761     5.3623     0.00

/usr/local/lib/python3.13/dist-packages/linearmodels/iv/model.py:789: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(


In [39]:
#table
models = {
    "(1)": iv_1_r,
    "(2)": iv_2_r,
    "(8)": iv_8_r,
    "(9)": iv_9_r,
}

rows = []
for var, label in labels.items():
    coef_row, se_row = [label], [""]
    for m in models.values():
        if var in m.params.index:
            coef_row.append(f"{m.params[var]:.2f}")
            se_row.append(f"({m.std_errors[var]:.2f})")
        else:
            coef_row.append("")
            se_row.append("")
    rows += [coef_row, se_row]

rows.append(["Dependent variable"] + ["Log GDP per capita, 1995"] * 3 + ["Log output per worker, 1988"])
rows.append(["Standard errors"] + ["Robust (HC0)"] * 4)
rows.append(["N"] + [f"{int(m.nobs)}" for m in models.values()])

table_b3_iv = pd.DataFrame(rows, columns=[""] + list(models))

print(table_b3_iv.to_latex(
    index=False,
    escape=False,
    caption="Two-stage least squares with heteroskedasticity-robust standard errors: Table 4, Panel A, columns (1), (2), (8), and (9)",
))

\begin{table}
\caption{Two-stage least squares with heteroskedasticity-robust standard errors: Table 4, Panel A, columns (1), (2), (8), and (9)}
\begin{tabular}{lllll}
\toprule
 & (1) & (2) & (8) & (9) \\
\midrule
Average protection against expropriation risk, 1985--1995 & 0.94 & 1.00 & 1.11 & 0.98 \\
 & (0.18) & (0.24) & (0.50) & (0.20) \\
Latitude &  & -0.65 & -1.18 &  \\
 &  & (1.23) & (1.79) &  \\
Asia dummy &  &  & -1.05 &  \\
 &  &  & (0.50) &  \\
Africa dummy &  &  & -0.44 &  \\
 &  &  & (0.38) &  \\
``Other'' continent dummy &  &  & -0.99 &  \\
 &  &  & (1.06) &  \\
Dependent variable & Log GDP per capita, 1995 & Log GDP per capita, 1995 & Log GDP per capita, 1995 & Log output per worker, 1988 \\
Standard errors & Robust (HC0) & Robust (HC0) & Robust (HC0) & Robust (HC0) \\
N & 64 & 64 & 64 & 61 \\
\bottomrule
\end{tabular}
\end{table}

